# Assignment Part 4 — Atomic Business Operations

Demonstrates 5 atomic operations:
1. `create_case()` — Create a surgical work order
2. `create_appointment()` — Book OR with atomic multi-resource reservation
3. `cancel_appointment()` — Release all resources atomically
4. `emergency_override()` — Preempt elective cases for emergencies
5. `complete_appointment()` — Mark surgery done, set equipment to sterilizing

Each operation either **fully succeeds** (COMMIT) or **fully fails** (ROLLBACK) — no partial state.

In [23]:
import sys
sys.path.insert(0, '../src')

from datetime import date, time, timedelta
from rich.console import Console
from rich.table import Table
from rich import print as rprint
console = Console()

from sqlalchemy import select, text
from sqlalchemy.orm import Session
from or_scheduler.database import engine, SessionLocal
from or_scheduler.models import Department, Staff, Room, Patient, Equipment, Appointment, AuditLog
from or_scheduler.operations import (
    create_case, create_appointment, cancel_appointment,
    emergency_override, complete_appointment,
    StaffItem, RoomConflictError, AppointmentStateError
)

In [24]:
"""
Reset transactional data so this notebook is idempotent (safe to re-run).
Reference data (departments, staff, rooms, equipment, patients, schedules) is preserved.
"""
from sqlalchemy import text as _text

with Session(engine) as _s:
    # Delete in strict FK order (children first, then parents)
    _s.execute(_text("DELETE FROM override_displaced_appointments"))
    _s.execute(_text("DELETE FROM overrides"))
    _s.execute(_text("DELETE FROM audit_log"))
    _s.execute(_text("DELETE FROM equipment_reservations"))
    _s.execute(_text("DELETE FROM staff_reservations"))
    _s.execute(_text("DELETE FROM room_reservations"))
    _s.execute(_text("DELETE FROM appointments"))
    _s.execute(_text("DELETE FROM cases"))
    _s.commit()

TARGET_DATE = date.today() + timedelta(days=5)
print(f"✅ Transactional tables cleared. Target date: {TARGET_DATE}")

✅ Transactional tables cleared. Target date: 2026-03-09


## Setup — Load Seed Data References

In [25]:
with Session(engine) as s:
    dept = s.execute(select(Department).limit(1)).scalar_one()
    surgeon = s.execute(select(Staff).where(Staff.role == 'SURGEON').limit(1)).scalar_one()
    anaest = s.execute(select(Staff).where(Staff.role == 'ANAESTHESIOLOGIST').limit(1)).scalar_one()
    scrub = s.execute(select(Staff).where(Staff.role == 'SCRUB_NURSE').limit(1)).scalar_one()
    room_or1 = s.execute(select(Room).where(Room.room_code == 'OR-3')).scalar_one()
    room_or2 = s.execute(select(Room).where(Room.room_code == 'OR-4')).scalar_one()
    patients = s.execute(select(Patient).limit(5)).scalars().all()
    equip = s.execute(select(Equipment).where(Equipment.serial_number == 'CARM-001')).scalar_one()
    
    # Capture IDs before session closes
    dept_id = dept.department_id
    surgeon_id = surgeon.staff_id
    anaest_id = anaest.staff_id
    scrub_id = scrub.staff_id
    room_or1_id = room_or1.room_id
    room_or2_id = room_or2.room_id
    patient_hns = [p.hn for p in patients]
    equip_id = equip.equipment_id

print(f"Surgeon: {surgeon.name}")
print(f"Anaesthesiologist: {anaest.name}")
print(f"Scrub Nurse: {scrub.name}")
print(f"Room: {room_or1.room_code}")
print(f"Target date: {TARGET_DATE}")

Surgeon: นพ.สมชาย วงศ์สุวรรณ
Anaesthesiologist: นพ.กิตติ บุญเสริม
Scrub Nurse: พยาบาล สุนิษา เพชรรัตน์
Room: OR-3
Target date: 2026-03-09


## Operation 1 — `create_case()`

In [26]:
session = SessionLocal()

case_result = create_case(
    session,
    patient_hn=patient_hns[0],
    department_id=dept_id,
    surgeon_id=surgeon_id,
    procedure_type='Total Knee Replacement',
    urgency='ELECTIVE',
    estimated_duration_minutes=120,
    clinical_notes='Patient has mild hypertension, no other comorbidities.',
    created_by=surgeon_id,
)
session.commit()

t = Table(title="Op 1: Case Created", show_header=True, header_style="bold green")
t.add_column("Field")
t.add_column("Value", style="yellow")
t.add_row("case_id", str(case_result.case_id))
t.add_row("status", case_result.status)
t.add_row("urgency", case_result.urgency)
t.add_row("procedure_type", case_result.procedure_type)
console.print(t)

case_id = case_result.case_id

                   Op 1: Case Created                    
┏━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Field          ┃ Value                                ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ case_id        │ 3b383018-0600-47da-8b62-d909f256ec27 │
│ status         │ OPEN                                 │
│ urgency        │ ELECTIVE                             │
│ procedure_type │ Total Knee Replacement               │
└────────────────┴──────────────────────────────────────┘

## Atomicity — The Problem: What if a Booking Fails Halfway?

Without atomic transactions, a failure midway through `create_appointment()` could leave corrupted partial data:
- An `appointments` row exists, but no `room_reservation` was created
- Staff reservations committed, but equipment reservation failed
- The OR appears unavailable (phantom booking) with no actual room lock

The code below proves this **cannot happen** — any failure causes a full rollback of all changes.
The database either reflects the complete operation or nothing at all.

In [27]:
import uuid as _uuid_mod

print("PROBLEM CASE: What if create_appointment() fails halfway through?")
print("Passing an INVALID staff ID to force a failure inside the operation...\n")

try:
    with SessionLocal() as _s:
        create_appointment(
            _s,
            case_id=case_id,            # valid case from Op 1
            room_id=room_or1_id,
            scheduled_date=TARGET_DATE,
            start_time=time(14, 0),     # unused time slot — no conflict
            end_time=time(16, 0),
            staff_items=[StaffItem(str(_uuid_mod.uuid4()), 'SURGEON')],  # ← INVALID UUID
            confirmed_by=surgeon_id,
        )
        _s.commit()  # never reached
except Exception as _e:
    print(f"  ✓ Operation rejected: {type(_e).__name__}")
    print(f"    {_e}\n")

# Verify: no partial records were left in the database
with engine.connect() as _c:
    _appt_count = _c.execute(text(
        "SELECT COUNT(*) FROM appointments WHERE case_id = :cid"
    ), {'cid': str(case_id)}).scalar()
    _res_count = _c.execute(text(
        "SELECT COUNT(*) FROM room_reservations rr "
        "JOIN appointments a ON a.appointment_id = rr.appointment_id "
        "WHERE a.case_id = :cid"
    ), {'cid': str(case_id)}).scalar()

print("DB state after failed booking:")
print(f"  appointments for this case     : {_appt_count}  (expected: 0)")
print(f"  room_reservations for this case: {_res_count}  (expected: 0)")
print()
print("✅ ATOMICITY CONFIRMED — zero partial records left.")
print("   The entire transaction rolled back. No half-booked state is possible.")

PROBLEM CASE: What if create_appointment() fails halfway through?
Passing an INVALID staff ID to force a failure inside the operation...

  ✓ Operation rejected: StaffNotFoundError
    Staff 19eb10a0-c123-4e4e-b686-b260c36afe6a not found.

DB state after failed booking:
  appointments for this case     : 0  (expected: 0)
  room_reservations for this case: 0  (expected: 0)

✅ ATOMICITY CONFIRMED — zero partial records left.
   The entire transaction rolled back. No half-booked state is possible.


## Operation 2 — `create_appointment()` (Solution: Happy Path)

With valid parameters, all 6 tables are updated atomically in a single transaction.

In [28]:
appt_result = create_appointment(
    session,
    case_id=case_id,
    room_id=room_or1_id,
    scheduled_date=TARGET_DATE,
    start_time=time(8, 0),
    end_time=time(10, 0),
    staff_items=[
        StaffItem(surgeon_id, 'SURGEON'),
        StaffItem(anaest_id, 'ANAESTHESIOLOGIST'),
        StaffItem(scrub_id, 'SCRUB_NURSE'),
    ],
    equipment_ids=[equip_id],
    confirmed_by=surgeon_id,
)
session.commit()

t = Table(title="Op 2: Appointment Booked", show_header=True, header_style="bold green")
t.add_column("Field")
t.add_column("Value", style="yellow")
t.add_row("appointment_id", str(appt_result.appointment_id))
t.add_row("status", appt_result.status)
t.add_row("version", str(appt_result.version))
t.add_row("date", str(appt_result.scheduled_date))
t.add_row("time", f"{appt_result.start_time}–{appt_result.end_time}")
t.add_row("room_reservation_id", str(appt_result.room_reservation_id))
t.add_row("staff_reservations", str(len(appt_result.staff_reservation_ids)))
t.add_row("equipment_reservations", str(len(appt_result.equipment_reservation_ids)))
console.print(t)

appt_id = appt_result.appointment_id

                    Op 2: Appointment Booked                     
┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Field                  ┃ Value                                ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ appointment_id         │ c36f7c7a-60a3-44b9-8587-e745f317b664 │
│ status                 │ CONFIRMED                            │
│ version                │ 1                                    │
│ date                   │ 2026-03-09                           │
│ time                   │ 08:00:00–10:00:00                    │
│ room_reservation_id    │ 7944e678-818d-4fb3-943a-9022b6eeb14b │
│ staff_reservations     │ 3                                    │
│ equipment_reservations │ 1                                    │
└────────────────────────┴──────────────────────────────────────┘

## Operation 2 — Double-Booking (Problem Case + System Response)

**The problem:** Two coordinators simultaneously try to book OR-3 for different patients at the same time. Without conflict detection, both could succeed — a patient safety disaster.

**What the system does:** The second booking attempt raises `RoomConflictError` and rolls back. After the error, verify the database to confirm only ONE reservation exists.

In [29]:
case2 = create_case(
    session, patient_hn=patient_hns[1], department_id=dept_id,
    surgeon_id=surgeon_id, procedure_type='Hip Replacement',
    created_by=surgeon_id
)
session.commit()

print("Attempting to book the SAME room at the SAME time (should raise RoomConflictError)...")
try:
    create_appointment(
        session,
        case_id=case2.case_id,
        room_id=room_or1_id,  # ← Same room
        scheduled_date=TARGET_DATE,
        start_time=time(8, 30),  # ← Overlaps with 08:00–10:00
        end_time=time(10, 30),
        staff_items=[StaffItem(surgeon_id, 'SURGEON')],
        confirmed_by=surgeon_id,
    )
    print("❌ ERROR: Double-booking was not prevented!")
except RoomConflictError as e:
    session.rollback()  # roll back the failed transaction
    print(f"✅ RoomConflictError correctly raised:")
    print(f"   {e}")

# Verify: only ONE reservation exists in the database
with engine.connect() as _c:
    _res_count = _c.execute(text("""
        SELECT COUNT(*) FROM room_reservations rr
        JOIN appointments a ON a.appointment_id = rr.appointment_id
        WHERE a.scheduled_date = :d
          AND rr.room_id = :rid
          AND rr.status NOT IN ('RELEASED', 'COMPLETED')
    """), {'d': str(TARGET_DATE), 'rid': str(room_or1_id)}).scalar()
print(f"\nDB verification:")
print(f"  Active reservations for OR-3 on {TARGET_DATE}: {_res_count}")
print(f"  ✅ Exactly 1 reservation — double-booking was prevented, data integrity maintained.")

Attempting to book the SAME room at the SAME time (should raise RoomConflictError)...
✅ RoomConflictError correctly raised:
   Room OR-3 is already booked from 08:00 to 10:00 on 2026-03-09.

DB verification:
  Active reservations for OR-3 on 2026-03-09: 1
  ✅ Exactly 1 reservation — double-booking was prevented, data integrity maintained.


## Operation 3 — `cancel_appointment()`

In [30]:
# Book a new appointment (different time) then cancel it
case3 = create_case(
    session, patient_hn=patient_hns[2], department_id=dept_id,
    surgeon_id=surgeon_id, procedure_type='Appendectomy',
    created_by=surgeon_id
)
appt3 = create_appointment(
    session,
    case_id=case3.case_id,
    room_id=room_or1_id,
    scheduled_date=TARGET_DATE,
    start_time=time(11, 0),
    end_time=time(12, 30),
    staff_items=[StaffItem(surgeon_id, 'SURGEON'), StaffItem(anaest_id, 'ANAESTHESIOLOGIST')],
    confirmed_by=surgeon_id,
)
session.commit()
print(f"Created appointment {appt3.appointment_id} (status: {appt3.status})")

# Now cancel it
cancel_appointment(
    session,
    appointment_id=appt3.appointment_id,
    cancelled_by=surgeon_id,
    reason='Patient condition changed — surgery not required at this time',
)
session.commit()

with Session(engine) as verify_s:
    cancelled = verify_s.execute(
        select(Appointment).where(Appointment.appointment_id == appt3.appointment_id)
    ).scalar_one()
    print(f"✅ Appointment status is now: {cancelled.status} (version: {cancelled.version})")

Created appointment 6c99dfc0-290d-484a-a17b-91b31d1cd5e4 (status: CONFIRMED)
✅ Appointment status is now: CANCELLED (version: 2)


## Operation 4 — `emergency_override()`

In [31]:
# Create an elective appointment that will be bumped
# NOTE: Staff are booked 8:00–10:00 from Op 2, so we use 10:30–12:00 here
elective_case = create_case(
    session, patient_hn=patient_hns[3], department_id=dept_id,
    surgeon_id=surgeon_id, procedure_type='Elective Shoulder Repair',
    created_by=surgeon_id
)
elective_appt = create_appointment(
    session,
    case_id=elective_case.case_id,
    room_id=room_or2_id,
    scheduled_date=TARGET_DATE,
    start_time=time(10, 30),
    end_time=time(12, 0),
    staff_items=[StaffItem(anaest_id, 'ANAESTHESIOLOGIST')],
    confirmed_by=surgeon_id,
)
session.commit()
print(f"Elective appointment {elective_appt.appointment_id} booked in OR-4 (10:30–12:00)")

# Emergency case preempts the same room/time
emg_case = create_case(
    session, patient_hn=patient_hns[4], department_id=dept_id,
    surgeon_id=surgeon_id, procedure_type='Ruptured Aortic Aneurysm',
    urgency='EMERGENCY', created_by=surgeon_id
)
override_result = emergency_override(
    session,
    case_id=emg_case.case_id,
    room_id=room_or2_id,
    scheduled_date=TARGET_DATE,
    start_time=time(10, 30),
    end_time=time(12, 0),
    staff_items=[
        StaffItem(surgeon_id, 'SURGEON'),
        StaffItem(scrub_id, 'SCRUB_NURSE'),
    ],
    authorized_by=surgeon_id,
    authorization_code='EMR-2026-001',
    override_reason='Ruptured aortic aneurysm — immediate surgical intervention required',
    clinical_urgency_score=1,
)
session.commit()

t = Table(title="Op 4: Emergency Override", show_header=True, header_style="bold red")
t.add_column("Field")
t.add_column("Value", style="yellow")
t.add_row("override_id", str(override_result.override_id))
t.add_row("emergency_appointment_id", str(override_result.emergency_appointment_id))
t.add_row("bumped_count", str(override_result.bumped_count))
t.add_row("displaced_ids", str(override_result.displaced_appointment_ids))
console.print(t)

with Session(engine) as verify_s:
    bumped = verify_s.execute(
        select(Appointment).where(Appointment.appointment_id == elective_appt.appointment_id)
    ).scalar_one()
    print(f"✅ Elective appointment is now: {bumped.status}")

Elective appointment dfa31f15-5432-4bc6-92b3-a0275b13c838 booked in OR-4 (10:30–12:00)


                          Op 4: Emergency Override                           
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Field                    ┃ Value                                          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ override_id              │ 5df6b1fe-3d42-463f-9a85-f60a37a4c5a3           │
│ emergency_appointment_id │ f1e6df9b-977a-44ac-b9e1-23a7c65244bc           │
│ bumped_count             │ 1                                              │
│ displaced_ids            │ [UUID('dfa31f15-5432-4bc6-92b3-a0275b13c838')] │
└──────────────────────────┴────────────────────────────────────────────────┘

✅ Elective appointment is now: BUMPED


## Operation 5 — `complete_appointment()`

In [32]:
from datetime import datetime, timezone

complete_appointment(
    session,
    appointment_id=appt_id,  # the first appointment we created
    completed_by=surgeon_id,
    actual_end_time=datetime.now(timezone.utc),
    notes='Surgery completed successfully. No complications.',
)
session.commit()

from or_scheduler.models import Equipment
with Session(engine) as verify_s:
    completed = verify_s.execute(
        select(Appointment).where(Appointment.appointment_id == appt_id)
    ).scalar_one()
    equipment = verify_s.execute(
        select(Equipment).where(Equipment.equipment_id == equip_id)
    ).scalar_one()
    print(f"✅ Appointment status: {completed.status}")
    print(f"✅ Equipment {equipment.serial_number} status: {equipment.status}")
    print(f"   Sterilization ends at: {equipment.last_sterilization_end}")

✅ Appointment status: COMPLETED
✅ Equipment CARM-001 status: STERILIZING
   Sterilization ends at: 2026-03-04 04:06:12.728185+00:00


## Audit Log — Complete Record of All Operations

In [33]:
session.close()

with Session(engine) as s:
    logs = s.execute(
        select(AuditLog).order_by(AuditLog.log_id.desc()).limit(20)
    ).scalars().all()

t = Table(title="Audit Log (last 20 entries)", show_header=True, header_style="bold magenta")
t.add_column("log_id", style="dim")
t.add_column("entity_type", style="cyan")
t.add_column("action", style="yellow")
t.add_column("old_status")
t.add_column("new_status", style="green")
t.add_column("changed_at")

for log in reversed(logs):
    t.add_row(
        str(log.log_id),
        log.entity_type,
        log.action,
        log.old_status or "-",
        log.new_status,
        log.changed_at.strftime("%H:%M:%S") if log.changed_at else "",
    )
console.print(t)

print("\n✅ All 5 atomic operations demonstrated successfully.")
print("   Every operation writes to audit_log with PostgreSQL transaction ID.")

                        Audit Log (last 20 entries)                        
┏━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ log_id ┃ entity_type ┃ action    ┃ old_status ┃ new_status ┃ changed_at ┃
┡━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ 75288  │ CASE        │ CREATED   │ -          │ OPEN       │ 00:18:21   │
│ 75289  │ APPOINTMENT │ CONFIRMED │ -          │ CONFIRMED  │ 00:18:21   │
│ 75290  │ CASE        │ CREATED   │ -          │ OPEN       │ 00:18:21   │
│ 75291  │ CASE        │ CREATED   │ -          │ OPEN       │ 00:18:21   │
│ 75292  │ APPOINTMENT │ CONFIRMED │ -          │ CONFIRMED  │ 00:18:21   │
│ 75293  │ APPOINTMENT │ CANCELLED │ CONFIRMED  │ CANCELLED  │ 00:18:21   │
│ 75294  │ CASE        │ CREATED   │ -          │ OPEN       │ 00:18:21   │
│ 75295  │ APPOINTMENT │ CONFIRMED │ -          │ CONFIRMED  │ 00:18:21   │
│ 75296  │ CASE        │ CREATED   │ -          │ OPEN       │ 00:18:21   │
│ 75297  │ APPOINTMENT │ BUMPED    │ CONFIRMED  │ BUMPED     │ 00:18:21   │
│ 75298  │ OVERRIDE    │ OVERRIDE  │ -          │ CONFIRMED  │ 00:18:21   │
│ 75299  │ APPOINTMENT │ COMPLETED │ CONFIRMED  │ COMPLETED  │ 00:18:21   │
└────────┴─────────────┴───────────┴────────────┴────────────┴────────────┘


✅ All 5 atomic operations demonstrated successfully.
   Every operation writes to audit_log with PostgreSQL transaction ID.
